In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def symbolic_dtft_two_params_with_phase():
    omega, omega_0 = sp.symbols('omega omega_0', real=True)
    alpha, beta = sp.symbols('alpha beta', real=True)
    n = sp.symbols('n', integer=True, nonnegative=True)

    # --- 1. SYMBOLIC DTFT CALCULATION ---
    term1 = (1 / (2 * sp.I)) * (alpha * sp.exp(sp.I * omega_0))**n
    term2 = (1 / (2 * sp.I)) * (alpha * sp.exp(-sp.I * omega_0))**n
    term3 = beta**n

    r1 = alpha * sp.exp(sp.I * omega_0) * sp.exp(-sp.I * omega)
    r2 = alpha * sp.exp(-sp.I * omega_0) * sp.exp(-sp.I * omega)
    r3 = beta * sp.exp(-sp.I * omega)

    sum1 = 1 / (1 - r1)
    sum2 = 1 / (1 - r2)
    sum3 = 1 / (1 - r3)

    X_omega_sym = (1 / (2 * sp.I)) * sum1 - (1 / (2 * sp.I)) * sum2 + sum3
    X_omega_simplified = sp.trigsimp(sp.factor(sp.together(X_omega_sym)))
    magnitude_sym = sp.simplify(sp.Abs(X_omega_simplified))

    print(r"--- Symbolically Derived Results ---")
    print(r"DTFT X(e^{j\omega}):")
    display(X_omega_simplified)
    print(r"Magnitude |X(e^{j\omega})|:")
    display(magnitude_sym)

    # Δημιουργούμε lambdify απευθείας για τη μιγαδική συνάρτηση ώστε να πάρουμε και πλάτος και φάση
    f_complex = sp.lambdify((omega, alpha, beta, omega_0), X_omega_simplified, modules='numpy')

    # --- 2. INTERACTIVE PLOTS ---
    def update_plots(alpha_val, beta_val, omega_0_val):
        clear_output(wait=True)
        omega_vals = np.linspace(-3*np.pi, 3*np.pi, 4000)
        
        # Υπολογισμός μιγαδικών τιμών
        X_vals = np.asarray(f_complex(omega_vals, alpha_val, beta_val, omega_0_val), dtype=complex)
        
        # Διαχωρισμός σε Πλάτος (Magnitude) και Φάση (Phase)
        mag_vals = np.abs(X_vals)
        phase_vals = np.unwrap(np.angle(X_vals))

        fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        
        # --- Magnitude Plot ---
        label_mag = r'Magnitude ($\alpha$=' + f'{alpha_val:.3f}, ' + r'$\beta$=' + f'{beta_val:.3f}, ' + r'$\omega_0$=' + f'{omega_0_val:.3f}' + r'$\pi$)'
        title_mag = r'DTFT Magnitude Spectrum ($\alpha$=' + f'{alpha_val:.3f}, ' + r'$\beta$=' + f'{beta_val:.3f}, ' + r'$\omega_0$=' + f'{omega_0_val:.3f}' + r'$\pi$)'

        axes[0].plot(omega_vals/np.pi, mag_vals, color='red', lw=2.5, label=label_mag)
        axes[0].set_title(title_mag, fontsize=12, fontweight='bold')
        axes[0].set_ylabel('Magnitude', fontsize=10)
        axes[0].grid(True, linestyle='--', alpha=0.6)
        axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
        
        # --- Phase Plot ---
        label_phase = r'Phase ($\alpha$=' + f'{alpha_val:.3f}, ' + r'$\beta$=' + f'{beta_val:.3f}, ' + r'$\omega_0$=' + f'{omega_0_val:.3f}' + r'$\pi$)'
        axes[1].plot(omega_vals/np.pi, phase_vals, color='blue', lw=2.5, label=label_phase)
        axes[1].set_title('DTFT Phase Spectrum', fontsize=12, fontweight='bold')
        axes[1].set_xlabel(r'Normalized Frequency $\omega / \pi$', fontsize=10)
        axes[1].set_ylabel('Phase (radians)', fontsize=10)
        axes[1].set_xlim(-3, 3)
        axes[1].set_xticks([-3, -2, -1, 0, 1, 2, 3])
        axes[1].set_xticklabels(['-3π', '-2π', '-π', '0', 'π', '2π', '3π'])
        axes[1].grid(True, linestyle='--', alpha=0.6)
        axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
        
        plt.subplots_adjust(left=0.08, right=0.80, top=0.93, bottom=0.09, hspace=0.30)
        plt.show()

        display(HTML("""
        <div style="border:1px solid #ccc; padding:12px; border-radius:5px; background-color:#f9f9f9; font-family:sans-serif; font-size:14px;">
        <strong>Symbolic DTFT:</strong> The complex spectrum was derived symbolically from first principles. The plotted expressions represent the magnitude and unwrapped phase evaluated numerically for the selected parameter values.
        </div>
        """))

    alpha_slider = widgets.FloatSlider(value=0.5, min=-0.95, max=0.95, step=0.05, description='Parameter alpha:', style={'description_width':'initial'})
    beta_slider = widgets.FloatSlider(value=0.4, min=-0.95, max=0.95, step=0.05, description='Parameter beta:', style={'description_width':'initial'})
    omega0_slider = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='omega_0 / pi:', style={'description_width':'initial'})

    ui = widgets.VBox([alpha_slider, beta_slider, omega0_slider])
    out = widgets.interactive_output(update_plots, {
        'alpha_val': alpha_slider, 
        'beta_val': beta_slider, 
        'omega_0_val': omega0_slider
    })
    
    display(ui, out)

symbolic_dtft_two_params_with_phase()